# Imports

In [1]:
from langchain_community.llms import LlamaCpp
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain.chains import RetrievalQA

# Config

In [2]:
GENERATION_MODEL_PATH = 'Phi-3-mini-4k-instruct-fp16.gguf'
RETREIVAL_MODEL_NAME = 'thenlper/gte-small'

In [3]:
FACTUAL_DATA = """
Interstellar is a 2014 epic science fiction film co-written, directed, and pro
duced by Christopher Nolan.
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin,
Ellen Burstyn, Matt Damon, and Michael Caine.
Set in a dystopian future where humanity is struggling to survive, the film
follows a group of astronauts who travel through a wormhole near Saturn in
search of a new home for mankind.
Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its
origins in a script Jonathan developed in 2007.
Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne
was an executive producer, acted as a scientific consultant, and wrote a tie-in
book, The Science of Interstellar.
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision
anamorphic format and IMAX 70 mm.
Principal photography began in late 2013 and took place in Alberta, Iceland,
and Los Angeles.
Interstellar uses extensive practical and miniature effects and the company
Double Negative created additional digital effects.
Interstellar premiered on October 26, 2014, in Los Angeles.
In the United States, it was first released on film stock, expanding to venues
using digital projectors.
The film had a worldwide gross over $677 million (and $773 million with subse
quent re-releases), making it the tenth-highest grossing film of 2014.
It received acclaim for its performances, direction, screenplay, musical score,
visual effects, ambition, themes, and emotional weight.
It has also received praise from many astronomers for its scientific accuracy
and portrayal of theoretical astrophysics. Since its premiere, Interstellar
gained a cult following,[5] and now is regarded by many sci-fi experts as one
of the best science-fiction films of all time.
Interstellar was nominated for five awards at the 87th Academy Awards, winning
Best Visual Effects, and received numerous other accolades"""

# Download Model

In [4]:
!wget --no-clobber "https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf"

File ‘Phi-3-mini-4k-instruct-fp16.gguf’ already there; not retrieving.



# RAG System

## Generation Model

In [5]:
llm = LlamaCpp(
    model_path=GENERATION_MODEL_PATH,
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False,
)

## Retrieval Model

In [6]:
texts = FACTUAL_DATA.split('.')
texts = [t.strip(' \n') for t in texts]

In [7]:
embedding_model = HuggingFaceEmbeddings(model_name=RETREIVAL_MODEL_NAME)

/tmp/ipykernel_9256/1561884496.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embedding_model = HuggingFaceEmbeddings(model_name=RETREIVAL_MODEL_NAME)
/home/mmahdi/.cache/pypoetry/virtualenvs/rag-example-FwGg7RSF-py3.10/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [8]:
retreival_db = FAISS.from_texts(texts, embedding_model)

## RAG Prompt

In [9]:
template = """<|user|>
Relevant information:
{context}
Provide a concise answer the following question using the relevant information
provided above:
{question}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"],
)

## RAG Pipeline

In [10]:
rag = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=retreival_db.as_retriever(),
    chain_type_kwargs={
        "prompt": prompt
    },
    verbose=True,
)

## Invoke

In [11]:
rag.invoke('What is the income generated by Interestellar?')



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'What is the income generated by Interestellar?',
 'result': ' The Income generated by Interstellar is over $677 million worldwide, with an additional $773 million from subsequent re-releases. This made it the tenth-highest grossing film of 2014. Moreover, its use of practical and digital effects contributed to this success.'}